## 04. Modeliranje: Eksperiment A (SC + classical FE + ML)

### Uvoz biblioteka

In [3]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

### Učitavanje podataka

In [4]:
DATA_DIR = "../../data/processed/function"
FEATURES_DIR = "../../data/features/function"
MODELS_DIR = "../../data/models/function"
METRICS_DIR = "../../results/metrics/function"

os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

RANDOM_STATE = 42

# Train/test split (Entry + enkodirana labela) 
train_entries_df = pd.read_csv(os.path.join(FEATURES_DIR, "sc_train_entries.csv"))
test_entries_df = pd.read_csv(os.path.join(FEATURES_DIR, "sc_test_entries.csv"))

entries_sc_train = train_entries_df["Entry"].values
entries_sc_test = test_entries_df["Entry"].values

y_sc_train = train_entries_df["label_encoded"].values
y_sc_test = test_entries_df["label_encoded"].values

# LabelEncoder fit-ovan jednom u 03 (nema ponovnog fitovanja)
le = joblib.load(os.path.join(FEATURES_DIR, "sc_label_encoder.pkl"))

print(f"Train skup: {len(entries_sc_train)} proteina")
print(f"Test skup:  {len(entries_sc_test)} proteina")
print(f"Klase: {list(le.classes_)}")

Train skup: 5330 proteina
Test skup:  1333 proteina
Klase: ['Hydrolase', 'Receptor', 'Structural protein', 'Transcription factor', 'Transport protein']


### Učitavanje raw obilježja

In [5]:
# AAC-CV je izračunat jednom za cijeli dataset (fiksni vokabular, ne zahtijeva fitovanje)
aac_df = pd.read_csv(os.path.join(FEATURES_DIR, "aac_cv_features.csv"), index_col="Entry")
aac_cols = list(aac_df.columns)

print(f"AAC-CV kolone: {len(aac_cols)}")
aac_df.head()

AAC-CV kolone: 20


,A,C,D,E,F,G,H,I,K,L,M,N,P,Q,R,S,T,V,W,Y
Entry,,,,,,,,,,,,,,,,,,,,
P03986,0.042328,0.031746,0.079365,0.052910,0.037037,0.026455,0.015873,0.074074,0.100529,0.105820,0.015873,0.063492,0.047619,0.031746,0.021164,0.058201,0.095238,0.042328,0.015873,0.042328
Q9NRU3,0.083070,0.021030,0.051525,0.070452,0.037855,0.079916,0.017876,0.028391,0.033649,0.126183,0.012618,0.032597,0.065195,0.030494,0.075710,0.075710,0.057834,0.065195,0.009464,0.025237
Q01167,0.110606,0.006061,0.025758,0.045455,0.021212,0.092424,0.028788,0.046970,0.042424,0.060606,0.012121,0.031818,0.107576,0.057576,0.051515,0.093939,0.068182,0.074242,0.004545,0.018182
P51788,0.102450,0.018931,0.028953,0.062361,0.052339,0.072383,0.017817,0.052339,0.035635,0.110245,0.025612,0.015590,0.060134,0.037862,0.065702,0.073497,0.059020,0.074610,0.015590,0.018931
Q9NXT0,0.042289,0.059701,0.017413,0.097015,0.037313,0.067164,0.074627,0.034826,0.064677,0.069652,0.007463,0.022388,0.032338,0.034826,0.099502,0.114428,0.047264,0.032338,0.004975,0.039801


In [6]:
# Sirove sekvence (fitovanje TF-IDF+SVD dešava se unutar Pipeline-a, po CV foldu)
sc_train_seq_df = pd.read_csv(os.path.join(FEATURES_DIR, "sc_train_sequences_raw.csv"), index_col="Entry")
sc_test_seq_df = pd.read_csv(os.path.join(FEATURES_DIR, "sc_test_sequences_raw.csv"), index_col="Entry")
seq_col = "Sequence"

print(f"Sirove sekvence — train: {sc_train_seq_df.shape[0]} | test: {sc_test_seq_df.shape[0]}")
sc_train_seq_df.head()

Sirove sekvence — train: 5330 | test: 1333


,Sequence
Entry,
Q9UKF2,MRSVQIFLSQCRLLLLLVPTMLLKSLGEDVIFHPEGEFDSYEVTIP...
P60953,MQTIKCVVVGDGAVGKTCLLISYTTNKFPSEYVPTVFDNYAVTVMI...
Q92985,MALAPERAAPRVLFGEWLLGEISSGCYEGLQWLDEARTCFRVPWKH...
Q8N131,MGLGARGAWAALLLGTLQVLALLGAAHESAAMAASANIENSGLPHN...
Q8NH79,MRNGTVITEFILLGFPVIQGLQTPLFIAIFLTYILTLAGNGLIIAT...


In [8]:
# Sirove (neskalirane) physchem osobine (StandardScaler fituje se unutar Pipeline-a, po CV foldu)
sc_train_ph_df = pd.read_csv(os.path.join(FEATURES_DIR, "sc_train_physchem_raw.csv"), index_col="Entry")
sc_test_ph_df = pd.read_csv(os.path.join(FEATURES_DIR, "sc_test_physchem_raw.csv"), index_col="Entry")
ph_cols = list(sc_train_ph_df.columns)

print(f"PH kolone: {ph_cols}")
sc_train_ph_df.head()

PH kolone: ['MW', 'pI', 'GRAVY', 'Aromaticity', 'Instability']


,MW,pI,GRAVY,Aromaticity,Instability
Entry,,,,,
Q9UKF2,88939.1664,7.842466,-0.354810,0.086076,45.503038
P60953,21258.3571,6.156766,-0.157068,0.083770,44.060733
Q92985,54276.9160,5.885871,-0.367197,0.077535,63.174155
Q8N131,21530.8606,8.763913,-0.116346,0.033654,30.125000
Q8NH79,34726.4299,9.097279,0.878526,0.115385,32.173109


### Sastavljanje DataFrame-a 

Spajamo AAC-CV, raw sekvence i neskalirane PH vrijednosti u jedan DataFrame po Entry-ju, poravnat sa redoslijedom labela (y_sc_train/y_sc_test).

In [9]:
X_train_raw = aac_df.join(sc_train_seq_df, how="inner").join(sc_train_ph_df, how="inner") 
X_test_raw = aac_df.join(sc_test_seq_df, how="inner").join(sc_test_ph_df, how="inner") 

# Poravnanje redoslijeda redova sa redoslijedom labela
X_train_raw = X_train_raw.loc[entries_sc_train]
X_test_raw = X_test_raw.loc[entries_sc_test]

assert list(X_train_raw.index) == list(entries_sc_train)
assert list(X_test_raw.index) == list(entries_sc_test)

print(f"X_train_raw: {X_train_raw.shape} | X_test_raw: {X_test_raw.shape}")
X_train_raw.head()

X_train_raw: (5330, 26) | X_test_raw: (1333, 26)


,A,C,D,E,F,G,H,I,K,L,...,T,V,W,Y,Sequence,MW,pI,GRAVY,Aromaticity,Instability
Entry,,,,,,,,,,,,,,,,,,,,,
Q9UKF2,0.040506,0.055696,0.043038,0.072152,0.039241,0.077215,0.027848,0.053165,0.072152,0.092405,...,0.037975,0.056962,0.011392,0.035443,MRSVQIFLSQCRLLLLLVPTMLLKSLGEDVIFHPEGEFDSYEVTIP...,88939.1664,7.842466,-0.354810,0.086076,45.503038
P60953,0.057592,0.031414,0.057592,0.068063,0.041885,0.047120,0.010471,0.041885,0.089005,0.099476,...,0.083770,0.099476,0.005236,0.036649,MQTIKCVVVGDGAVGKTCLLISYTTNKFPSEYVPTVFDNYAVTVMI...,21258.3571,6.156766,-0.157068,0.083770,44.060733
Q92985,0.111332,0.027833,0.041750,0.069583,0.031809,0.093439,0.019881,0.011928,0.029821,0.107356,...,0.047714,0.049702,0.023857,0.021869,MALAPERAAPRVLFGEWLLGEISSGCYEGLQWLDEARTCFRVPWKH...,54276.9160,5.885871,-0.367197,0.077535,63.174155
Q8N131,0.096154,0.004808,0.024038,0.028846,0.009615,0.067308,0.033654,0.048077,0.038462,0.072115,...,0.168269,0.062500,0.004808,0.019231,MGLGARGAWAALLLGTLQVLALLGAAHESAAMAASANIENSGLPHN...,21530.8606,8.763913,-0.116346,0.033654,30.125000
Q8NH79,0.057692,0.038462,0.009615,0.019231,0.073718,0.051282,0.025641,0.105769,0.035256,0.147436,...,0.099359,0.070513,0.009615,0.032051,MRNGTVITEFILLGFPVIQGLQTPLFIAIFLTYILTLAGNGLIIAT...,34726.4299,9.097279,0.878526,0.115385,32.173109


### Definisanje feature-set kombinacija (kroz ColumnTransfer)

In [10]:
# Funkcija koja bira odgovarajuci transformator i vraca Pipeline
# (svaka feature-set/model kombinacija ima sopstveni transformator)
def make_tfidf_svd_step():
    return Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(2, 3), max_features=3000, lowercase=False)),
        ("svd", TruncatedSVD(n_components=150, random_state=RANDOM_STATE)),
    ])

# Funkcija koja vraca ColumnTransfer za odgovarajucu feature-set kombinaciju
def make_preprocessor(feature_set_name):
    if feature_set_name == "AAC-CV":
        transformers = [("aac", "passthrough", aac_cols)]
    elif feature_set_name == "TFIDF-SVD":
        transformers = [("tfidf-svd", make_tfidf_svd_step(), seq_col)]
    elif feature_set_name == "PH":
        transformers = [("ph_scaled", StandardScaler(), ph_cols)]
    elif feature_set_name == "AAC-CV+TFIDF-SVD":
        transformers = [
            ("aac", "passthrough", aac_cols),
            ("tfidf-svd", make_tfidf_svd_step(), seq_col),
        ]
    elif feature_set_name == "AAC-CV+PH":
        transformers = [
            ("aac", "passthrough", aac_cols),
            ("ph_scaled", StandardScaler(), ph_cols),
        ]
    elif feature_set_name == "TFIDF-SVD+PH":
        transformers = [
            ("tfidf-svd", make_tfidf_svd_step(), seq_col),
            ("ph_scaled", StandardScaler(), ph_cols),
        ]
    elif feature_set_name == "AAC-CV+TFIDF-SVD+PH":
        transformers = [
            ("aac", "passthrough", aac_cols),
            ("tfidf-svd", make_tfidf_svd_step(), seq_col),
            ("ph_scaled", StandardScaler(), ph_cols),
        ]
    else:
        raise ValueError(f"Nepoznat feature set: {feature_set_name}")

    return ColumnTransformer(transformers=transformers, remainder="drop")

FEATURE_SET_NAMES = [
    "AAC-CV",
    "TFIDF-SVD",
    "PH",
    "AAC-CV+TFIDF-SVD",
    "AAC-CV+PH",
    "TFIDF-SVD+PH",
    "AAC-CV+TFIDF-SVD+PH",
]


### Definisanje modela i mreže hiperparametara

In [11]:
param_grids = {
    "LogisticRegression": {
        "model": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
        "params": {
            "C": [0.01, 0.1, 1, 10, 100],
            "solver": ["lbfgs"],
        },
    },
    "RandomForest": {
        "model": RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        "params": {
            "n_estimators": [200],
            "max_depth": [10, 15],
            "min_samples_leaf": [2, 5],
        },
    },
    "SVM": {
        "model": SVC(class_weight="balanced", random_state=RANDOM_STATE),
        "params": {
            "C": [0.1, 1, 10],
            "kernel": ["rbf", "linear"],
        },
    },
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

### Glavna petlja treniranja

In [12]:
from joblib import Memory

cache_dir = "../cache/pipeline_cache"
memory = Memory(location=cache_dir, verbose=0)

all_results = []

for feature_name in FEATURE_SET_NAMES:
    preprocessor = make_preprocessor(feature_name)

    for model_name, config in param_grids.items():
        print(f"Treniranje: {model_name:<20} | Feature set: {feature_name}")

        full_pipeline = Pipeline([
            ("features", preprocessor),
            ("model", config["model"]),
        ], memory=memory)

        pipeline_param_grid = {
            f"model__{param_name}": values 
            for param_name, values in config["params"].items()
        }

        grid = GridSearchCV(
            estimator=full_pipeline,
            param_grid=pipeline_param_grid,
            cv=cv,
            scoring="f1_weighted",
            n_jobs=-1,
            refit=True,
        )

        grid.fit(X_train_raw, y_sc_train)

        best_pipeline = grid.best_estimator_

        # Predikcije na train skupu (isti fit-ovan model, bez dodatnog treniranja)
        y_train_pred = best_pipeline.predict(X_train_raw)
        train_acc = accuracy_score(y_sc_train, y_train_pred)
        train_f1 = f1_score(y_sc_train, y_train_pred, average="weighted")

        # Predikcije na test skupu
        y_test_pred = best_pipeline.predict(X_test_raw)
        test_acc = accuracy_score(y_sc_test, y_test_pred)
        test_f1 = f1_score(y_sc_test, y_test_pred, average="weighted")

        best_params = {
            k.replace("model__", ""): v for k, v in grid.best_params_.items()
        }

        all_results.append({
            "Feature set": feature_name,
            "Model": model_name,
            "Best params": best_params,

            "CV F1 (weighted)": grid.best_score_,

            "Train Accuracy": train_acc,
            "Train F1 (weighted)": train_f1,

            "Test Accuracy": test_acc,
            "Test F1 (weighted)": test_f1,
            
            "Overfit Gap (F1)": train_f1 - test_f1,
        })

        model_filename = f"sc_{model_name}_{feature_name}.pkl"
        joblib.dump(best_pipeline, os.path.join(MODELS_DIR, model_filename))

    memory.clear()

print("\nTreniranje završeno.")

Treniranje: LogisticRegression   | Feature set: AAC-CV
Treniranje: RandomForest         | Feature set: AAC-CV
Treniranje: SVM                  | Feature set: AAC-CV


[Memory(location=../cache/pipeline_cache\joblib)]: Flushing completely the cache


Treniranje: LogisticRegression   | Feature set: TFIDF-SVD
Treniranje: RandomForest         | Feature set: TFIDF-SVD
Treniranje: SVM                  | Feature set: TFIDF-SVD


[Memory(location=../cache/pipeline_cache\joblib)]: Flushing completely the cache


Treniranje: LogisticRegression   | Feature set: PH
Treniranje: RandomForest         | Feature set: PH
Treniranje: SVM                  | Feature set: PH


...

Treniranje: LogisticRegression   | Feature set: TFIDF-SVD+PH
Treniranje: RandomForest         | Feature set: TFIDF-SVD+PH
Treniranje: SVM                  | Feature set: TFIDF-SVD+PH


[Memory(location=../cache/pipeline_cache\joblib)]: Flushing completely the cache


Treniranje: LogisticRegression   | Feature set: AAC-CV+TFIDF-SVD+PH
Treniranje: RandomForest         | Feature set: AAC-CV+TFIDF-SVD+PH
Treniranje: SVM                  | Feature set: AAC-CV+TFIDF-SVD+PH


[Memory(location=../cache/pipeline_cache\joblib)]: Flushing completely the cache



Treniranje završeno.


### Pregled i čuvanje rezultata

In [13]:
results_df = pd.DataFrame(all_results)

# Sortiranje po CV F1, uz uvid u Overfit Gap
results_df = results_df.sort_values(by="CV F1 (weighted)", ascending=False).reset_index(drop=True)
results_df.to_csv(os.path.join(METRICS_DIR, "sc_classical_modeling_results.csv"), index=False)

display_cols = [
    "Model", "Feature set",
    "Train Accuracy", "Train F1 (weighted)",
    "CV F1 (weighted)",
    "Test Accuracy", "Test F1 (weighted)",
    "Overfit Gap (F1)",
]

results_df[display_cols].style.format({
    "Train Accuracy": "{:.3f}",
    "Train F1 (weighted)": "{:.3f}",
    "CV F1 (weighted)": "{:.3f}",
    "Test Accuracy": "{:.3f}",
    "Test F1 (weighted)": "{:.3f}",
    "Overfit Gap (F1)": "{:.3f}",
}).background_gradient(subset=["Overfit Gap (F1)"], cmap="Reds")

,Model,Feature set,Train Accuracy,Train F1 (weighted),CV F1 (weighted),Test Accuracy,Test F1 (weighted),Overfit Gap (F1)
0,SVM,AAC-CV+TFIDF-SVD,0.989,0.989,0.808,0.813,0.810,0.179
1,SVM,TFIDF-SVD,0.992,0.992,0.807,0.817,0.813,0.179
2,SVM,TFIDF-SVD+PH,0.918,0.918,0.788,0.804,0.803,0.115
3,SVM,AAC-CV+TFIDF-SVD+PH,0.920,0.920,0.788,0.802,0.801,0.119
4,SVM,AAC-CV,0.859,0.858,0.763,0.759,0.759,0.100
5,RandomForest,AAC-CV+TFIDF-SVD+PH,0.970,0.970,0.761,0.757,0.753,0.217
6,RandomForest,AAC-CV+PH,0.989,0.990,0.759,0.767,0.764,0.226
7,LogisticRegression,AAC-CV+TFIDF-SVD+PH,0.794,0.796,0.757,0.753,0.756,0.040
8,LogisticRegression,TFIDF-SVD+PH,0.791,0.793,0.756,0.752,0.755,0.039
9,RandomForest,TFIDF-SVD+PH,0.961,0.961,0.750,0.750,0.745,0.216
